# Notebook 03 — Prédiction du vainqueur (Deep Learning — MLP)

**Objectif** : Prédire le vainqueur d'une course F1 avant son départ  
**Architecture** : MLP (réseau dense) avec PyTorch — 3 couches cachées, dropout, batch normalization  
**Features** : position de départ, historique pilote/circuit, performance récente, météo, équipe  
**Évaluation** : Top-1 et Top-3 Accuracy, courbe d'apprentissage


In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from src.data_loader import load_session
from src.deep_learning import (
    build_race_dataset,
    prepare_Xy,
    train_mlp,
    top_k_accuracy,
    plot_learning_curve,
    plot_top_predictions,
    WinnerMLP,
    HAS_TORCH,
)

plt.rcParams['figure.dpi'] = 120
print(f'PyTorch disponible : {HAS_TORCH}')

if HAS_TORCH:
    import torch
    print(f'Version PyTorch : {torch.__version__}')
    print(f'CUDA disponible : {torch.cuda.is_available()}')

## 1. Chargement des sessions

In [ ]:
# 2025 inclus — sessions non encore disponibles silencieusement ignorées
YEARS = [2022, 2023, 2024, 2025]
GP_NAMES = ['Bahrain', 'Monaco', 'British', 'Italian', 'Abu Dhabi', 'Belgian', 'Hungarian']

sessions = []
for year in YEARS:
    for gp in GP_NAMES:
        s = load_session(year, gp, 'R')
        if s is not None:
            sessions.append(s)

print(f'{len(sessions)} sessions chargées')

In [ ]:
df = build_race_dataset(sessions)
print(f'Dataset : {len(df)} entrées pilote/course')
print(f'Vainqueurs : {df["IsWinner"].sum()}  ({100*df["IsWinner"].mean():.1f}%)')
df.head(10)

## 2. Exploration des features

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

features = ['GridPosition', 'AvgRecentPos', 'StdRecentPos', 'AvgCircuitPos', 'Rainfall', 'Points']
features = [f for f in features if f in df.columns]

for i, feat in enumerate(features[:6]):
    for winner, grp in df.groupby('IsWinner'):
        label = 'Vainqueur' if winner else 'Autres'
        color = '#f39c12' if winner else 'steelblue'
        axes[i].hist(grp[feat].dropna(), bins=25, alpha=0.6, label=label, color=color, density=True)
    axes[i].set_title(feat)
    axes[i].legend(fontsize=8)
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Distribution des features — Vainqueur vs Autres', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Taux de victoire par position de grille
if 'GridPosition' in df.columns:
    win_by_grid = df[df['GridPosition'] <= 10].groupby('GridPosition')['IsWinner'].mean() * 100
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(win_by_grid.index, win_by_grid.values, color='steelblue', alpha=0.8)
    ax.set_xlabel('Position sur la grille')
    ax.set_ylabel('Taux de victoire (%)')
    ax.set_title('Taux de victoire par position sur la grille (Top 10)', fontweight='bold')
    ax.grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

## 3. Préparation & split des données

In [ ]:
X, y, feature_cols = prepare_Xy(df)
print(f'Features ({len(feature_cols)}) : {feature_cols}')
print(f'Shape X : {X.shape}  |  Shape y : {y.shape}')

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, np.arange(len(df)), test_size=0.2, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.15, random_state=42)

print(f'Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}')

## 4. Architecture du réseau MLP

In [ ]:
if HAS_TORCH:
    model_preview = WinnerMLP(input_dim=len(feature_cols))
    print(model_preview)
    total_params = sum(p.numel() for p in model_preview.parameters())
    print(f'\nNombre de paramètres : {total_params:,}')
else:
    print('PyTorch non disponible — installez-le avec : pip install torch')

## 5. Entraînement

In [ ]:
EPOCHS = 50  # Augmentez pour de meilleurs résultats (ex: 100)

model, history = train_mlp(
    X_train, y_train,
    X_val, y_val,
    epochs=EPOCHS,
    lr=1e-3,
    batch_size=32,
)

## 6. Courbe d'apprentissage

In [ ]:
if history['train_loss']:
    fig = plot_learning_curve(history)
    plt.show()
else:
    print('Historique vide (PyTorch absent ?)')

## 7. Évaluation — Top-K Accuracy

In [ ]:
df_test = df.iloc[idx_test].reset_index(drop=True)

top1 = top_k_accuracy(model, X_test, df_test, k=1)
top3 = top_k_accuracy(model, X_test, df_test, k=3)
top5 = top_k_accuracy(model, X_test, df_test, k=5)

print(f'Top-1 Accuracy : {top1:.3f} ({top1*100:.1f}%)')
print(f'Top-3 Accuracy : {top3:.3f} ({top3*100:.1f}%)')
print(f'Top-5 Accuracy : {top5:.3f} ({top5*100:.1f}%)')

# Baseline : toujours prédire le pole comme vainqueur
baseline_top1 = (df_test[df_test['GridPosition'] == 1.0]['IsWinner'] == 1).mean() if 'GridPosition' in df_test.columns else 0
print(f'\nBaseline (pole = vainqueur) : {baseline_top1:.3f}')

In [ ]:
# Barplot de comparaison
fig, ax = plt.subplots(figsize=(8, 5))
labels = ['Top-1', 'Top-3', 'Top-5', 'Baseline']
values = [top1, top3, top5, baseline_top1]
colors = ['#e74c3c', '#3498db', '#2ecc71', '#95a5a6']
bars = ax.bar(labels, values, color=colors, alpha=0.85, edgecolor='white')
ax.set_ylim(0, 1.1)
ax.set_ylabel('Accuracy')
ax.set_title('Précision du modèle MLP (Top-K)', fontweight='bold')
ax.grid(True, axis='y', alpha=0.3)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.show()

## 8. Prédictions sur une course spécifique

In [ ]:
# Visualise les probabilités de victoire pour la première course du test set
fig = plot_top_predictions(model, X_test, df_test, race_idx=0)
plt.show()

In [ ]:
# Résultats détaillés par course
if HAS_TORCH and model is not None:
    import torch
    device = next(model.parameters()).device
    model.eval()
    with torch.no_grad():
        probs = model.predict_proba(torch.from_numpy(X_test).to(device)).cpu().numpy().flatten()
    
    df_test_pred = df_test.copy()
    df_test_pred['ProbWin'] = probs
    
    print('Top-3 pilotes prédits par course :')
    for (year, circuit), grp in df_test_pred.groupby(['Year', 'Circuit']):
        top3 = grp.nlargest(3, 'ProbWin')[['Driver', 'ProbWin', 'IsWinner']]
        winner_flag = '✓' if grp[grp['IsWinner']==1]['Driver'].iloc[0] in top3['Driver'].values else '✗'
        print(f'  {year} {circuit} {winner_flag}')
        for _, row in top3.iterrows():
            mark = '<- vainqueur réel' if row['IsWinner'] else ''
            print(f'    {row["Driver"]}: {row["ProbWin"]:.2%}  {mark}')